In [2]:
# ==============================================================
# 🇲🇦 Prétraitement Dataset - Moroccan Darija
# À exécuter AVANT le script de finetuning
# Compatible: Google Colab
# ==============================================================

# ============================================================
# ÉTAPE 0 — Installation
# ============================================================
# !pip install -q pandas datasets transformers sentencepiece

import os
import re
import pandas as pd
from datasets import Dataset
from collections import Counter
from google.colab import drive

# ============================================================
# ÉTAPE 1 — Monter Google Drive
# ============================================================
drive.mount('/content/drive')

DATASET_PATH  = "/content/drive/MyDrive/data/dataset_conversation.csv"
OUTPUT_PATH   = "/content/drive/MyDrive/data/dataset_clean.csv"
REPORT_PATH   = "/content/drive/MyDrive/data/preprocessing_report.txt"

# ============================================================
# ÉTAPE 2 — Chargement du dataset brut
# ============================================================
print("=" * 60)
print("📂 CHARGEMENT DU DATASET")
print("=" * 60)

df = pd.read_csv(DATASET_PATH)

print(f"✅ Lignes totales     : {len(df)}")
print(f"   Colonnes           : {list(df.columns)}")
print(f"   Types              :\n{df.dtypes}")
print(f"\n📌 Aperçu:")
print(df.head(3).to_string())

# ============================================================
# ÉTAPE 3 — Analyse initiale (avant nettoyage)
# ============================================================
print("\n" + "=" * 60)
print("🔍 ANALYSE INITIALE")
print("=" * 60)

print(f"\n📊 Valeurs nulles par colonne:")
print(df.isnull().sum())

print(f"\n📊 Distribution des langues:")
if 'language' in df.columns:
    print(df['language'].value_counts())

print(f"\n📊 Distribution des topics (top 10):")
if 'topic' in df.columns:
    print(df['topic'].value_counts().head(10))

print(f"\n📊 Distribution des sources:")
if 'source_file' in df.columns:
    print(df['source_file'].value_counts())

# Longueurs des textes
df['text_length'] = df['text'].astype(str).apply(len)
print(f"\n📊 Longueurs des textes (caractères):")
print(f"   Min    : {df['text_length'].min()}")
print(f"   Max    : {df['text_length'].max()}")
print(f"   Moyenne: {df['text_length'].mean():.0f}")
print(f"   Médiane: {df['text_length'].median():.0f}")

# ============================================================
# ÉTAPE 4 — Fonctions de nettoyage
# ============================================================
print("\n" + "=" * 60)
print("🧹 NETTOYAGE DU DATASET")
print("=" * 60)

# ---------- 4.1 Suppression des doublons ----------
before = len(df)
df = df.drop_duplicates(subset=['text'])
after = len(df)
print(f"\n[1] Doublons supprimés         : {before - after} ({before} → {after})")

# ---------- 4.2 Suppression des lignes nulles ----------
before = len(df)
df = df.dropna(subset=['text'])
df = df[df['text'].astype(str).str.strip() != '']
after = len(df)
print(f"[2] Lignes vides/null supprimées: {before - after} ({before} → {after})")

# ---------- 4.3 Vérification du format ChatML ----------
def has_valid_chatml(text: str) -> bool:
    """Vérifie que le texte contient bien le format <|im_start|>...<|im_end|>"""
    return (
        '<|im_start|>' in text and
        '<|im_end|>' in text and
        '<|im_start|>system' in text and
        '<|im_start|>user' in text and
        '<|im_start|>assistant' in text
    )

before = len(df)
df['valid_chatml'] = df['text'].astype(str).apply(has_valid_chatml)
invalid = df[~df['valid_chatml']]
print(f"[3] Format ChatML invalide     : {len(invalid)} lignes")
if len(invalid) > 0:
    print(f"    Exemples invalides:")
    print(invalid['text'].head(2).to_string())
df = df[df['valid_chatml']].drop(columns=['valid_chatml'])
print(f"    Après filtre               : {len(df)} lignes")

# ---------- 4.4 Nettoyage du texte ----------
def clean_text(text: str) -> str:
    """Nettoie le texte en préservant la structure ChatML"""
    # Supprimer les espaces/newlines excessifs entre les balises
    text = re.sub(r'\n{3,}', '\n', text)          # Max 2 newlines consécutifs
    text = re.sub(r'[ \t]{2,}', ' ', text)        # Espaces multiples → 1 espace
    text = re.sub(r'\r\n', '\n', text)             # Windows line endings
    text = re.sub(r'\r', '\n', text)               # Mac line endings
    # Supprimer les caractères de contrôle invisibles (sauf newline/tab)
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', text)
    return text.strip()

df['text'] = df['text'].astype(str).apply(clean_text)
print(f"[4] Nettoyage du texte         : ✅ appliqué")

# ---------- 4.5 Filtre longueur minimum ----------
MIN_LENGTH = 50    # Caractères minimum (évite les échanges vides)
MAX_LENGTH = 8000  # Caractères maximum (évite les textes trop longs)

before = len(df)
df['text_length'] = df['text'].apply(len)
df = df[(df['text_length'] >= MIN_LENGTH) & (df['text_length'] <= MAX_LENGTH)]
after = len(df)
print(f"[5] Filtre longueur [{MIN_LENGTH}-{MAX_LENGTH}]   : {before - after} supprimés ({before} → {after})")

# ---------- 4.6 Vérification assistant non vide ----------
def has_non_empty_assistant(text: str) -> bool:
    """Vérifie que la réponse assistant n'est pas vide"""
    parts = text.split('<|im_start|>assistant')
    if len(parts) < 2:
        return False
    assistant_part = parts[-1].replace('<|im_end|>', '').strip()
    return len(assistant_part) > 10

before = len(df)
df = df[df['text'].apply(has_non_empty_assistant)]
after = len(df)
print(f"[6] Réponses assistant vides   : {before - after} supprimés ({before} → {after})")

# ---------- 4.7 Extraction des tours de conversation ----------
def count_turns(text: str) -> int:
    """Compte le nombre de tours user/assistant"""
    return text.count('<|im_start|>user')

df['num_turns'] = df['text'].apply(count_turns)
print(f"\n[7] Distribution des tours de conversation:")
print(df['num_turns'].value_counts().sort_index().to_string())

# ---------- 4.8 Nettoyage des colonnes metadata ----------
if 'topic' in df.columns:
    df['topic'] = df['topic'].astype(str).str.strip()
if 'language' in df.columns:
    df['language'] = df['language'].astype(str).str.strip().str.lower()

# ============================================================
# ÉTAPE 5 — Formatage final pour le modèle
# ============================================================
print("\n" + "=" * 60)
print("🎯 FORMATAGE FINAL POUR LE MODÈLE")
print("=" * 60)

def format_for_training(text: str, eos_token: str = "<|endoftext|>") -> str:
    """
    Assure que le texte se termine correctement avec l'EOS token.
    Le tokenizer ajoutera son propre EOS token lors du chargement,
    mais on l'ajoute ici en prévention si absent.
    """
    text = text.rstrip()
    # S'assurer que le dernier <|im_end|> est présent
    if not text.endswith('<|im_end|>'):
        text = text + '<|im_end|>'
    return text

df['text'] = df['text'].apply(format_for_training)
print("✅ Format final appliqué (EOS token, trailing <|im_end|>)")

# ============================================================
# ÉTAPE 6 — Statistiques finales
# ============================================================
print("\n" + "=" * 60)
print("📊 STATISTIQUES FINALES")
print("=" * 60)

df['text_length'] = df['text'].apply(len)
df['num_tokens_approx'] = df['text_length'] // 4  # Approximation: ~4 chars/token

print(f"\n✅ Dataset final           : {len(df)} exemples")
print(f"\n📏 Longueurs (caractères):")
print(f"   Min       : {df['text_length'].min()}")
print(f"   Max       : {df['text_length'].max()}")
print(f"   Moyenne   : {df['text_length'].mean():.0f}")
print(f"   Médiane   : {df['text_length'].median():.0f}")

print(f"\n🔢 Tokens approximatifs (chars/4):")
print(f"   Total     : ~{df['num_tokens_approx'].sum():,} tokens")
print(f"   Moyenne   : ~{df['num_tokens_approx'].mean():.0f} tokens/exemple")

if 'language' in df.columns:
    print(f"\n🌍 Langues:")
    print(df['language'].value_counts().to_string())

if 'num_turns' in df.columns:
    print(f"\n💬 Tours de conversation:")
    print(df['num_turns'].value_counts().sort_index().to_string())

# ============================================================
# ÉTAPE 7 — Sauvegarde
# ============================================================
print("\n" + "=" * 60)
print("💾 SAUVEGARDE")
print("=" * 60)

# Colonnes à garder pour l'entraînement
cols_to_save = ['text', 'topic', 'language', 'source_file']
cols_to_save = [c for c in cols_to_save if c in df.columns]

df_save = df[cols_to_save].copy()
df_save.to_csv(OUTPUT_PATH, index=False, encoding='utf-8-sig')
print(f"✅ Dataset propre sauvegardé : {OUTPUT_PATH}")
print(f"   {len(df_save)} exemples | {len(cols_to_save)} colonnes")

# Rapport texte
report_lines = [
    "=" * 60,
    "RAPPORT DE PRÉTRAITEMENT — DARIJA DATASET",
    "=" * 60,
    f"Dataset source    : {DATASET_PATH}",
    f"Dataset sortie    : {OUTPUT_PATH}",
    f"Exemples finaux   : {len(df_save)}",
    f"Longueur moyenne  : {df['text_length'].mean():.0f} caractères",
    f"Tokens totaux     : ~{df['num_tokens_approx'].sum():,}",
    "",
    "Filtres appliqués :",
    "  - Suppression doublons",
    "  - Suppression lignes null/vides",
    "  - Vérification format ChatML",
    "  - Nettoyage caractères spéciaux",
    f"  - Filtre longueur [{MIN_LENGTH}-{MAX_LENGTH}] chars",
    "  - Vérification réponse assistant non vide",
]

with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))
print(f"✅ Rapport sauvegardé        : {REPORT_PATH}")

# ============================================================
# ÉTAPE 8 — Aperçu final + vérification HuggingFace Dataset
# ============================================================
print("\n" + "=" * 60)
print("🔎 VÉRIFICATION COMPATIBILITÉ HUGGINGFACE DATASETS")
print("=" * 60)

hf_dataset = Dataset.from_pandas(df_save.reset_index(drop=True))
print(f"✅ HuggingFace Dataset OK : {hf_dataset}")

# Afficher un exemple formaté
print(f"\n📌 Exemple d'entrée pour le modèle:")
print("-" * 40)
sample = df_save['text'].iloc[0]
print(sample[:500] + "..." if len(sample) > 500 else sample)
print("-" * 40)

print("\n🎉 Prétraitement terminé!")
print(f"   👉 Utilise maintenant '{OUTPUT_PATH}' comme DATASET_PATH dans le script de finetuning.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📂 CHARGEMENT DU DATASET
✅ Lignes totales     : 10544
   Colonnes           : ['text', 'topic', 'language', 'source_file']
   Types              :
text           object
topic          object
language       object
source_file    object
dtype: object

📌 Aperçu:
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    

In [4]:
# ==============================================================
# 🌍 Finetuning MULTILINGUE — Tout le dataset sans filtre langue
# Modèle: Qwen2.5-0.5B-Instruct
# Compatible: Google Colab (GPU T4/A100)
# ==============================================================

# ============================================================
# CELLULE 1 — Installation (une seule fois)
# ============================================================
# !pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps trl peft accelerate bitsandbytes

# ============================================================
# CELLULE 2 — Entraînement
# ============================================================
import unsloth  # ✅ TOUJOURS en premier
import os
import torch
import pandas as pd
from datasets import Dataset
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel
from google.colab import drive

# ============================================================
# ÉTAPE 1 — Monter Google Drive
# ============================================================
drive.mount('/content/drive')

# ============================================================
# ÉTAPE 2 — Configuration
# ============================================================
MODEL_NAME     = "Qwen/Qwen2.5-0.5B-Instruct"
MAX_SEQ_LENGTH = 2048
DTYPE          = None
LOAD_IN_4BIT   = True

# 📁 Chemins
DATASET_PATH = "/content/drive/MyDrive/data/dataset_conversation.csv"
OUTPUT_DIR   = "/content/drive/MyDrive/models/multilingual_finetuned"
LOGS_DIR     = "/content/drive/MyDrive/models/multilingual_logs"

# 🏋️ Paramètres
NUM_EPOCHS       = 3
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 4
LEARNING_RATE    = 2e-4
WARMUP_STEPS     = 10
MAX_STEPS        = -1
SAVE_STEPS       = 50
LOGGING_STEPS    = 10

# 🎯 LoRA
LORA_R         = 16
LORA_ALPHA     = 16
LORA_DROPOUT   = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]

# ============================================================
# ÉTAPE 3 — Chargement du modèle
# ============================================================
print("📦 Chargement du modèle...")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name    = MODEL_NAME,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = DTYPE,
    load_in_4bit  = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = LORA_R,
    target_modules             = TARGET_MODULES,
    lora_alpha                 = LORA_ALPHA,
    lora_dropout               = LORA_DROPOUT,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
    use_rslora                 = False,
    loftq_config               = None,
)

print(f"✅ Modèle chargé: {MODEL_NAME}")
model.print_trainable_parameters()

# ============================================================
# ÉTAPE 4 — Chargement du dataset COMPLET (toutes langues)
# ============================================================
print("\n📂 Chargement du dataset complet...")

df = pd.read_csv(DATASET_PATH)
print(f"✅ Total lignes brutes: {len(df)}")

# Afficher la distribution des langues (juste pour info, pas de filtre)
if 'language' in df.columns:
    print("\n📊 Distribution des langues dans le dataset:")
    print(df['language'].value_counts())
    print("👉 Toutes les langues seront utilisées pour l'entraînement")

if 'topic' in df.columns:
    print(f"\n📊 Nombre de topics uniques: {df['topic'].nunique()}")

# ============================================================
# ÉTAPE 5 — Nettoyage MINIMAL (sans filtre de langue)
# ============================================================
before = len(df)

# Uniquement: supprimer les lignes vraiment vides ou nulles
df = df.dropna(subset=['text'])
df = df[df['text'].astype(str).str.strip() != '']

# Supprimer les doublons exacts
df = df.drop_duplicates(subset=['text'])

after = len(df)
print(f"\n✅ Après nettoyage minimal: {after} exemples ({before - after} supprimés)")
print(f"   ✅ Aucun filtre de langue appliqué — dataset 100% multilingue")

# ============================================================
# ÉTAPE 6 — Formatage (EOS token uniquement)
# ============================================================
EOS_TOKEN = tokenizer.eos_token

def format_sample(example):
    text = str(example['text'])
    if not text.endswith(EOS_TOKEN):
        text = text + EOS_TOKEN
    return {"text": text}

dataset = Dataset.from_pandas(df[['text']].reset_index(drop=True))
dataset = dataset.map(format_sample)

# Split 90/10
split         = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset  = split['test']

print(f"\n✅ Split dataset:")
print(f"   Train : {len(train_dataset)} exemples")
print(f"   Eval  : {len(eval_dataset)} exemples")

# ============================================================
# ÉTAPE 7 — TrainingArguments
# ============================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,   exist_ok=True)

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    warmup_steps                = WARMUP_STEPS,
    max_steps                   = MAX_STEPS,
    learning_rate               = LEARNING_RATE,
    fp16                        = not torch.cuda.is_bf16_supported(),
    bf16                        = torch.cuda.is_bf16_supported(),
    logging_steps               = LOGGING_STEPS,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    eval_strategy               = "steps",
    eval_steps                  = SAVE_STEPS,
    load_best_model_at_end      = True,
    optim                       = "adamw_8bit",
    weight_decay                = 0.01,
    lr_scheduler_type           = "linear",
    seed                        = 42,
    report_to                   = "none",
    logging_dir                 = LOGS_DIR,
    dataloader_num_workers      = 0,
)

# ============================================================
# ÉTAPE 8 — Trainer
# ============================================================
trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_dataset,
    eval_dataset       = eval_dataset,
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args               = training_args,
)

# ============================================================
# ÉTAPE 9 — Entraînement
# ============================================================
print("\n🚀 Début de l'entraînement multilingue...")

gpu_stats        = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
max_memory       = round(gpu_stats.total_memory / 1024**3, 3)
print(f"   GPU: {gpu_stats.name} | VRAM: {max_memory} GB | Utilisé: {start_gpu_memory} GB")

trainer_stats = trainer.train()

end_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"\n✅ Entraînement terminé!")
print(f"   Temps total  : {trainer_stats.metrics['train_runtime']:.2f}s")
print(f"   Loss finale  : {trainer_stats.metrics['train_loss']:.4f}")
print(f"   VRAM utilisée: {end_gpu_memory} GB / {max_memory} GB")

# ============================================================
# ÉTAPE 10 — Sauvegarde LoRA + merge complet
# ============================================================
print(f"\n💾 Sauvegarde des adapters LoRA dans {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Adapters LoRA sauvegardés!")

# ✅ Sauvegarde modèle mergé (résout le bug d'inférence Unsloth)
MERGED_DIR = "/content/drive/MyDrive/models/multilingual_merged_final"
print(f"\n💾 Sauvegarde modèle mergé dans {MERGED_DIR}...")
os.makedirs(MERGED_DIR, exist_ok=True)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("✅ Modèle mergé sauvegardé — utilise ce dossier pour l'inférence!")



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ HuggingFace login OK
📦 Chargement de Llama-3.2-1B-Instruct...
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.10G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...


✅ Entraînement terminé!
   Temps     : 5439.57s
   Loss      : 0.9926
   VRAM      : 3.045 GB / 14.563 GB

💾 Sauvegarde dans /content/drive/MyDrive/models/llama1b_darija...


Both `max_new_tokens` (=256) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ Adapters LoRA sauvegardés!

🧪 Test du modèle...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i

RuntimeError: output with shape [1, 32, 1, 64] doesn't match the broadcast shape [1, 32, 58, 64]

### Test de génération avec le modèle entraîné

In [9]:
# ============================================================
# ÉTAPE 10 — Sauvegarde LoRA + merge complet
# ============================================================
print(f"\n💾 Sauvegarde des adapters LoRA dans {OUTPUT_DIR}...")
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("✅ Adapters LoRA sauvegardés!")

# ✅ Sauvegarde modèle mergé (résout le bug d'inférence Unsloth)
MERGED_DIR = "/content/drive/MyDrive/models/multilingual_merged_final"
print(f"\n💾 Sauvegarde modèle mergé dans {MERGED_DIR}...")
os.makedirs(MERGED_DIR, exist_ok=True)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print("✅ Modèle mergé sauvegardé — utilise ce dossier pour l'inférence!")

# ============================================================
# ÉTAPE 11 — Test depuis le modèle MERGÉ (sans bug Unsloth)
# ============================================================
print("\n🧪 Test depuis le modèle mergé...")

import unsloth # Added import
# No need for AutoModelForCausalLM, AutoTokenizer as unsloth handles it
import torch

# Load model and tokenizer using unsloth's FastLanguageModel for merged models
model_test, tokenizer_test = unsloth.FastLanguageModel.from_pretrained(
    model_name    = MERGED_DIR,
    max_seq_length= MAX_SEQ_LENGTH,
    dtype         = torch.float16, # Assuming merged_16bit means float16
    load_in_4bit  = False, # It's a merged 16bit model, no need for 4bit loading here.
)
# model_test.eval() # unsloth.FastLanguageModel handles this internally




💾 Sauvegarde des adapters LoRA dans /content/drive/MyDrive/models/llama1b_darija...
✅ Adapters LoRA sauvegardés!

💾 Sauvegarde modèle mergé dans /content/drive/MyDrive/models/multilingual_merged_final...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 599.19it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:21<00:00, 81.92s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/models/multilingual_merged_final`
✅ Modèle mergé sauvegardé — utilise ce dossier pour l'inférence!

🧪 Test depuis le modèle mergé...
==((====))==  Unsloth 2026.3.8: Fast Llama patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

The tokenizer you are loading from '/content/drive/MyDrive/models/multilingual_merged_final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
The tokenizer you are loading from '/content/drive/MyDrive/models/multilingual_merged_final' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Unsloth: Will load /content/drive/MyDrive/models/multilingual_merged_final as a legacy tokenizer.
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for


🇲🇦 Darija:
   👤 User     : كيفاش نحمي حسابي ديالي فالإنترنت؟
   🤖 Assistant: إيه يا عزيزي؟ أنا بخير، أنا بخير. كيف يمكنني أن أساعدك اليوم؟


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🇫🇷 Français:
   👤 User     : Comment protéger mon compte en ligne?
   🤖 Assistant: Pour sécuriser votre compte en ligne, commence par choisir des mots de passe qui sont longs (au moins 12 caractères), complexes et changés régulièrement.

🇬🇧 Anglais:
   👤 User     : How can I protect my online account?
   🤖 Assistant: To keep your accounts safe online, make sure they have strong, unique passwords that include numbers, symbols, and letters.

🎉 Entraînement et test multilingue terminés!
   👉 Modèle final : /content/drive/MyDrive/models/multilingual_merged_final


In [14]:
# Test 1 — Darija
messages_1 = [
    {"role": "system", "content": "نت مساعد مفيد ووديد. كتهضر بالدارجة المغربية."},
    {"role": "user",   "content": "سلام لباس"}
]

# Test 2 — Français
messages_2 = [
    {"role": "system", "content": "Tu es un assistant utile."},
    {"role": "user",   "content": "c'est quoi la capitale du l'Algérie?"}
]

# Test 3 — Anglais
messages_3 = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "How can I protect my online account?"}
]

for i, messages in enumerate([messages_1, messages_2, messages_3], 1):
    input_text = tokenizer_test.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer_test(input_text, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model_test.generate(
            **inputs,
            max_new_tokens = 150,
            temperature    = 0.7,
            top_p          = 0.9,
            do_sample      = True,
            pad_token_id   = tokenizer_test.eos_token_id,
        )

    response = tokenizer_test.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    lang = ["🇲🇦 Darija", "🇫🇷 Français", "🇬🇧 Anglais"][i-1]
    print(f"\n{lang}:")
    print(f"   👤 User     : {messages[-1]['content']}")
    print(f"   🤖 Assistant: {response}")

print("\n🎉 Entraînement et test multilingue terminés!")
print(f"   👉 Modèle final : {MERGED_DIR}")

Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🇲🇦 Darija:
   👤 User     : سلام لباس
   🤖 Assistant: بسم الله الرحمن الرحيم، أهلا بيك في فندق جايدي. شكراً لبعض الوقت.


Both `max_new_tokens` (=150) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🇫🇷 Français:
   👤 User     : c'est quoi la capitale du l'Algérie?
   🤖 Assistant: la capitale du L'Algérie est Algérie.

🇬🇧 Anglais:
   👤 User     : How can I protect my online account?
   🤖 Assistant: To ensure your privacy online, make sure all information you share on the internet is secure.

🎉 Entraînement et test multilingue terminés!
   👉 Modèle final : /content/drive/MyDrive/models/multilingual_merged_final
